# RAG pipeline prototype — raw PDF → Qdrant (in-memory) → search → eval

Notebook **self-contained** (chạy được local & Google Colab), là **lab đo chất lượng retrieval**, bám flow trong `docs/decide/technique/`:

```
raw PDF → MarkItDown (parse) → canonical artifact → section split (heading)
        → caption (LLM, ý-nghĩa-nén) → embed(caption) → Qdrant in-memory
        → search (cùng embedder) → eval recall@k / MRR trên OpenRAGBench
```

**Scope (đã thống nhất):**
- ✅ Chốt được: caption-only vs content-embed (**D7**), chọn embedding model/dimension (**Day-0 §10**), score threshold / no-answer, recall@k, latency.
- ❌ KHÔNG chốt được ở đây (cần backend thật + concurrency): atomic claim/race, durable queue, write-order, fail-closed, deploy — *in-memory không chứng minh production semantics* (LESSONS §2).

> Đây là phòng lab, không phải eval gate. Khi số liệu ổn → trích golden queries + tiêu chí pass/fail thành pytest chạy trong CI (DAY0 §13), rồi ghi kết quả vào `NEW_REPO_DECISIONS.md`.

## 0. Cài đặt (Colab + local)

In [ ]:
%pip install -q "markitdown[all]" qdrant-client openai pandas huggingface_hub

## 1. Config — OpenAI-compatible SDK (model / url / api)

In [ ]:
import os, hashlib, uuid, time, re, json, random, shutil
from pathlib import Path
from collections import defaultdict

# ── Embedding: ingest + search PHẢI cùng model/dimension (search.md §2) ──
# Không set URL (hoặc set rỗng) → None → SDK mặc định gửi về OpenAI.
EMBED_BASE_URL = os.getenv("EMBED_BASE_URL") or None
EMBED_API_KEY  = os.getenv("EMBED_API_KEY", os.getenv("OPENAI_API_KEY", ""))
EMBED_MODEL    = os.getenv("EMBED_MODEL", "text-embedding-3-small")

# ── Caption LLM: ý-nghĩa-nén của section (ingestion.md §6) ──
CAPTION_BASE_URL = os.getenv("CAPTION_BASE_URL") or EMBED_BASE_URL   # None → OpenAI
CAPTION_API_KEY  = os.getenv("CAPTION_API_KEY") or EMBED_API_KEY
CAPTION_MODEL    = os.getenv("CAPTION_MODEL", "gpt-4o-mini")

# Trên Colab có thể set trực tiếp:  EMBED_API_KEY = "sk-..."
# Dùng provider OpenAI-compatible khác (OpenRouter/vLLM...): set EMBED_BASE_URL.

# ── Pipeline knobs ──
CAPTION_ENABLED   = True    # True = embed caption (flow chuẩn). False = embed content (baseline D7)
SECTION_MAX_CHARS = 4000    # length guard → sub-split (ingestion.md §5)
EMBED_BATCH       = 64
SEARCH_TOP_K      = 10

# ── Eval dataset ──
REPO_ID         = "gunnybd01/OpenRAGBench"
SAMPLE_N        = 20
SAMPLE_STRATEGY = "balanced"   # balanced | random | hard

assert EMBED_API_KEY, "Set EMBED_API_KEY (hoặc OPENAI_API_KEY) trước khi chạy"
print("embed:", EMBED_MODEL, "@", EMBED_BASE_URL or "api.openai.com (default)",
      "| caption:", CAPTION_MODEL, "| caption_enabled:", CAPTION_ENABLED)

## 2. Loader OpenRAGBench (inline — không cần file ngoài)

Sample N eval docs + distractors tỉ lệ tương ứng từ HuggingFace, tạo folder cùng kiến trúc gốc.

In [ ]:
from huggingface_hub import hf_hub_download

def _load_jsonl(fn):
    p = hf_hub_download(repo_id=REPO_ID, filename=fn, repo_type="dataset")
    with open(p) as f:
        return [json.loads(l) for l in f]

def _load_manifest():
    p = hf_hub_download(repo_id=REPO_ID, filename="eval/eval_manifest.json", repo_type="dataset")
    with open(p) as f:
        return json.load(f)

def download_sample_pdfs(N=20, strategy="balanced", seed=42, local_dir=None):
    random.seed(seed)
    print("Loading metadata from HuggingFace...")
    manifest    = _load_manifest()
    all_queries = _load_jsonl("metadata/eval_queries.jsonl")
    corpus_meta = _load_jsonl("metadata/corpus_metadata.jsonl")

    all_pairs  = manifest["eval_pairs"]
    total_docs = manifest["eval_docs"]
    total_dist = manifest["distractor_docs"]
    N_docs     = min(N, total_docs)

    doc_to_pairs = defaultdict(list)
    for p in all_pairs:
        doc_to_pairs[p["gt_doc_id"]].append(p)
    ids = list(doc_to_pairs.keys())

    if strategy == "balanced":
        ext = [d for d in ids if any(p["type"] == "extractive"  for p in doc_to_pairs[d])]
        ab  = [d for d in ids if any(p["type"] == "abstractive" for p in doc_to_pairs[d])]
        n_ext = round(N_docs * len(ext) / len(ids)); n_ab = N_docs - n_ext
        sampled = random.sample(ext, min(n_ext, len(ext))) + random.sample(ab, min(n_ab, len(ab)))
    elif strategy == "random":
        sampled = random.sample(ids, N_docs)
    elif strategy == "hard":
        ab  = [d for d in ids if any(p["type"] == "abstractive" for p in doc_to_pairs[d])]
        ext = [d for d in ids if d not in set(ab)]
        n_ab = min(N_docs, len(ab))
        sampled = random.sample(ab, n_ab) + random.sample(ext, N_docs - n_ab)
    else:
        raise ValueError("strategy: balanced | random | hard")
    random.shuffle(sampled); sset = set(sampled)
    pairs = [p for d in sampled for p in doc_to_pairs[d]]

    out = Path(local_dir) if local_dir else Path(f"./eval_sample_{N_docs}")
    if out.exists():
        shutil.rmtree(out)
    EVAL_DIR = out / "corpus" / "eval_docs"; DIST_DIR = out / "corpus" / "distractor_docs"
    for d in (EVAL_DIR, DIST_DIR):
        d.mkdir(parents=True, exist_ok=True)

    def _pull(hf_path, dst_dir, flat_name):
        hf_hub_download(repo_id=REPO_ID, filename=hf_path, repo_type="dataset",
                        local_dir=str(dst_dir), local_dir_use_symlinks=False)
        nested = dst_dir / hf_path; final = dst_dir / flat_name
        if nested.exists() and not final.exists():
            shutil.move(str(nested), str(final))

    print(f"Downloading {len(sampled)} eval PDFs...")
    got_eval = []
    for doc_id in sampled:
        try:
            _pull(f"corpus/eval_docs/{doc_id}.pdf", EVAL_DIR, f"{doc_id}.pdf"); got_eval.append(doc_id)
        except Exception as e:
            print("  fail", doc_id, e)

    n_dist = round(total_dist * N_docs / total_docs)
    dist = [m for m in corpus_meta if not m.get("is_eval_doc")
            and m["pdf_filename"].replace(".pdf", "") not in sset]
    dist = random.sample(dist, min(n_dist, len(dist)))
    print(f"Downloading {len(dist)} distractor PDFs...")
    got_dist = []
    for m in dist:
        try:
            _pull(f"corpus/distractor_docs/{m['pdf_filename']}", DIST_DIR, m["pdf_filename"])
            got_dist.append(m["pdf_filename"])
        except Exception as e:
            print("  fail distractor", m["pdf_filename"], e)

    print(f"eval={len(got_eval)} distractor={len(got_dist)} queries={len(pairs)}")
    return {"eval_pairs": pairs, "eval_docs_path": str(EVAL_DIR), "distractor_path": str(DIST_DIR)}

sample = download_sample_pdfs(N=SAMPLE_N, strategy=SAMPLE_STRATEGY)
EVAL_PAIRS  = sample["eval_pairs"]
corpus_pdfs = (sorted(Path(sample["eval_docs_path"]).glob("*.pdf"))
               + sorted(Path(sample["distractor_path"]).glob("*.pdf")))
print(f"Corpus PDFs: {len(corpus_pdfs)} | eval pairs: {len(EVAL_PAIRS)}")

## 3. Parser — MarkItDown → canonical artifact

- `document_id` deterministic theo **địa chỉ nguồn** (= stem PDF = `gt_doc_id`), KHÔNG hash markdown (parser.md §6).
- Guard **size TRƯỚC khi đọc** (CONSTRAINTS §2).
- Artifact (Markdown) là source-of-truth cho downstream — ở đây giữ in-memory thay cho S3.

In [ ]:
from markitdown import MarkItDown

NAMESPACE        = uuid.UUID("12345678-1234-5678-1234-567812345678")
MAX_SOURCE_BYTES = 50 * 1024 * 1024

_md       = MarkItDown()
ARTIFACTS = {}   # document_id -> markdown

def document_id_for(path):
    return path.stem   # khớp gt_doc_id

def parse_to_artifact(path):
    if path.stat().st_size > MAX_SOURCE_BYTES:
        raise ValueError(f"source too large: {path.stat().st_size} bytes")
    doc_id = document_id_for(path)
    ARTIFACTS[doc_id] = _md.convert(str(path)).text_content
    return doc_id

## 4. Section split — đơn vị NGHĨA (heading), không token-chunk

`section_id = f(document_id, order)` deterministic; giữ `heading_path` + `content_hash`; length guard → sub-split (ingestion.md §5).

In [ ]:
HEADING_RE = re.compile(r"^(#{1,6})\s+(.*)$")

def _subsplit(text, limit):
    if len(text) <= limit:
        return [text]
    parts, buf, size = [], [], 0
    for para in text.split("\n\n"):
        if size + len(para) > limit and buf:
            parts.append("\n\n".join(buf)); buf, size = [], 0
        buf.append(para); size += len(para)
    if buf:
        parts.append("\n\n".join(buf))
    return parts

def split_sections(doc_id):
    md_text = ARTIFACTS[doc_id]
    raw, cur, hpath = [], [], []
    def flush():
        body = "\n".join(cur).strip()
        if body:
            raw.append((list(hpath), body))
    for ln in md_text.splitlines():
        m = HEADING_RE.match(ln)
        if m:
            flush(); cur.clear()
            level = len(m.group(1))
            hpath = hpath[:level - 1] + [m.group(2).strip()]
        else:
            cur.append(ln)
    flush()
    if not raw:                      # không có heading → 1 section toàn doc
        raw = [([doc_id], md_text.strip())]
    out, order = [], 0
    for hp, body in raw:
        for chunk in _subsplit(body, SECTION_MAX_CHARS):
            out.append({
                "section_id":    f"{doc_id}::sec{order:04d}",
                "document_id":   doc_id,
                "heading_path":  hp,
                "section_order": order,
                "content":       chunk,
                "content_hash":  hashlib.sha256(chunk.encode()).hexdigest(),
            })
            order += 1
    return out

## 5. Caption + Embedding (OpenAI SDK) + cache theo content_hash

- Caption = nén ý nghĩa section (khử vocabulary mismatch tại indexing time).
- Coalescer-cache theo `content_hash` để skip re-embed (ingestion.md §6).
- `CAPTION_ENABLED=False` → embed thẳng content (baseline so sánh **D7**).

In [ ]:
from openai import OpenAI

emb_client = OpenAI(base_url=EMBED_BASE_URL, api_key=EMBED_API_KEY)
cap_client = OpenAI(base_url=CAPTION_BASE_URL, api_key=CAPTION_API_KEY)

# Cache giữ qua các lần re-run cell (KHÔNG reset → không mất caption đã tốn tiền sinh)
try:
    _caption_cache, _embed_cache
except NameError:
    _caption_cache, _embed_cache = {}, {}

CAPTION_PROMPT = (
    "Bạn nén ý nghĩa của một đoạn tài liệu thành 1-2 câu, tập trung vào CHỦ ĐỀ "
    "và CÁC THỰC THỂ/THUẬT NGỮ chính để phục vụ semantic search. "
    "Chỉ trả về câu tóm tắt, không thêm lời dẫn."
)

def _safe(t):
    """Không bao giờ trả chuỗi rỗng cho API embedding."""
    t = (t or "").strip()
    return t if t else "(no content)"

def make_caption(sec):
    h = sec["content_hash"]
    if h in _caption_cache:
        return _caption_cache[h]
    if not CAPTION_ENABLED:
        cap = sec["content"][:600]
    else:
        try:
            r = cap_client.chat.completions.create(
                model=CAPTION_MODEL, temperature=0,
                messages=[{"role": "system", "content": CAPTION_PROMPT},
                          {"role": "user", "content": sec["content"][:6000]}])
            cap = (r.choices[0].message.content or "").strip()
        except Exception as e:
            print("  caption fail:", e); cap = ""
        if not cap:                       # caption rỗng → fallback content (đừng để rỗng đi embed)
            cap = sec["content"][:600].strip()
    _caption_cache[h] = _safe(cap)
    return _caption_cache[h]

def embed_texts(items, max_retry=5):
    """items: list[(key, text)] -> dict key->vector. Cache theo key; retry+backoff; chống lệch vector."""
    todo = [(k, t) for k, t in items if k not in _embed_cache]
    for i in range(0, len(todo), EMBED_BATCH):
        batch = todo[i:i + EMBED_BATCH]
        texts = [_safe(t) for _, t in batch]          # sanitize: không có chuỗi rỗng
        for attempt in range(max_retry):
            try:
                r = emb_client.embeddings.create(model=EMBED_MODEL, input=texts)
                if len(r.data) != len(batch):
                    raise ValueError(f"embed count mismatch: got {len(r.data)} want {len(batch)}")
                break
            except Exception as e:
                if attempt == max_retry - 1:
                    raise
                wait = 2 ** attempt + random.random()
                print(f"  embed retry {attempt+1} (batch@{i}): {e} -> sleep {wait:.1f}s")
                time.sleep(wait)
        for (k, _), d in zip(batch, sorted(r.data, key=lambda x: x.index)):  # sort index chống lệch
            _embed_cache[k] = d.embedding
    return {k: _embed_cache[k] for k, _ in items}

## 6. Qdrant in-memory — collection ENCODE dimension

Probe dimension thật từ model rồi mới tạo collection (dimension ↔ index binding, CONSTRAINTS §2).

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

qdrant = QdrantClient(":memory:")

_probe = emb_client.embeddings.create(model=EMBED_MODEL, input=["probe"]).data[0].embedding
EMBED_DIMENSION = len(_probe)
COLLECTION = f"sections_dim{EMBED_DIMENSION}"          # index id encode dimension
qdrant.recreate_collection(
    COLLECTION, vectors_config=VectorParams(size=EMBED_DIMENSION, distance=Distance.COSINE))
print(f"Collection {COLLECTION} (dim={EMBED_DIMENSION})")

## 7. Ingest — vector = embedding(caption), payload = full content + lineage

Lineage `nguồn → artifact → section → vector` được giữ trong payload (search.md §6).

In [ ]:
def point_id(section_id):
    return str(uuid.uuid5(NAMESPACE, section_id))

t0, all_sections = time.time(), []
for pdf in corpus_pdfs:
    try:
        doc_id = parse_to_artifact(pdf)
    except Exception as e:
        print("parse fail", pdf.name, e); continue
    for s in split_sections(doc_id):
        s["source_uri"]   = str(pdf)
        s["artifact_uri"] = f"artifact://{doc_id}"
        s["caption"]      = make_caption(s)
        all_sections.append(s)
print(f"sections: {len(all_sections)}  (parse+caption {time.time()-t0:.1f}s)")

vecs = embed_texts([(s["content_hash"], s["caption"]) for s in all_sections])
points = [PointStruct(
    id=point_id(s["section_id"]),
    vector=vecs[s["content_hash"]],
    payload={
        "unit_id": s["section_id"], "document_id": s["document_id"],
        "display_name": " / ".join(s["heading_path"]) or s["document_id"],
        "caption": s["caption"], "content": s["content"],
        "heading_path": s["heading_path"],
        "lineage": {"artifact_uri": s["artifact_uri"], "source_uri": s["source_uri"]},
    }) for s in all_sections]
qdrant.upsert(COLLECTION, points=points)
print("upserted", len(points))

## 8. Search — response schema là CONTRACT

Cùng embedder với ingest; trả full content + cả hai lineage URI + score + correlation_id (search.md §6).

In [ ]:
def search(query, top_k=SEARCH_TOP_K, correlation_id=None):
    correlation_id = correlation_id or str(uuid.uuid4())
    qv = list(embed_texts([("q::" + hashlib.sha256(query.encode()).hexdigest(), query)]).values())[0]
    out = []
    for h in qdrant.search(COLLECTION, query_vector=qv, limit=top_k):
        p = h.payload
        out.append({
            "correlation_id": correlation_id,
            "unit_id": p["unit_id"], "document_id": p["document_id"],
            "display_name": p["display_name"], "caption": p["caption"],
            "content": p["content"], "heading_path": p["heading_path"],
            "lineage": p["lineage"], "score": h.score,
        })
    return out

demo = search(EVAL_PAIRS[0]["query"], top_k=5)
print("Q:", EVAL_PAIRS[0]["query"][:90])
print("gt:", EVAL_PAIRS[0]["gt_doc_id"])
for r in demo[:3]:
    print(f"  {r['score']:.3f}  {r['document_id']}  {r['display_name'][:50]}")

## 9. Eval — recall@k / MRR (doc-level) + latency

Hit = `gt_doc_id` xuất hiện trong các document_id trả về (distinct, giữ rank).

In [ ]:
def eval_recall(pairs, ks=(1, 3, 5, 10)):
    maxk = max(ks)
    hits = {k: 0 for k in ks}; rr = 0.0; n = 0; lat = []
    by_type = defaultdict(lambda: {"n": 0, **{k: 0 for k in ks}})
    for p in pairs:
        gt = p["gt_doc_id"]; t = time.time()
        res = search(p["query"], top_k=maxk); lat.append(time.time() - t)
        seen, ranked = set(), []
        for r in res:
            if r["document_id"] not in seen:
                seen.add(r["document_id"]); ranked.append(r["document_id"])
        rank = ranked.index(gt) + 1 if gt in ranked else None
        n += 1; tp = p.get("type", "?"); by_type[tp]["n"] += 1
        if rank:
            rr += 1.0 / rank
            for k in ks:
                if rank <= k:
                    hits[k] += 1; by_type[tp][k] += 1
    lat.sort()
    print(f"n={n}  MRR={rr/n:.3f}  p50={lat[len(lat)//2]*1000:.0f}ms  p95={lat[int(len(lat)*0.95)]*1000:.0f}ms")
    for k in ks:
        print(f"  recall@{k} = {hits[k]/n:.3f}")
    print("by type:")
    for tp, d in by_type.items():
        print(f"  {tp}: n={d['n']} " + " ".join(f"r@{k}={d[k]/d['n']:.2f}" for k in ks))

eval_recall(EVAL_PAIRS)

## 10. Next — biến số liệu thành quyết định

1. **D7 (caption-only vs content):** chạy lại với `CAPTION_ENABLED=False`, so recall@k. Caption thua content nhiều → cần hybrid/rerank (search.md §4).
2. **Dimension/model (Day-0 §10):** đổi `EMBED_MODEL`, so recall ↔ latency ↔ cost.
3. **No-answer / threshold:** thêm ngưỡng `score` tối thiểu, đo empty-rate vs precision.
4. Ghi số liệu + người chốt vào `docs/decide/NEW_REPO_DECISIONS.md` (PROPOSED → RATIFIED).